In [1]:
!pip install pandas scikit-learn joblib openpyxl


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

## import file


In [2]:
df=pd.read_csv('data.csv')
df.head(2)

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1


In [3]:
df['category'].value_counts(normalize=True)


category
 1    0.424978
 0    0.352815
-1    0.222207
Name: proportion, dtype: float64

In [4]:
df.columns

Index(['clean_comment', 'category'], dtype='object')

In [5]:
df.isnull().sum()

clean_comment    100
category           0
dtype: int64

In [6]:
df.dropna(subset=['clean_comment'], inplace=True)

In [7]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
37244    False
37245    False
37246    False
37247    False
37248    False
Length: 37149, dtype: bool

In [8]:
df['clean_comment'].isnull().sum()

np.int64(0)

In [9]:
df['clean_comment'].duplicated().sum()


np.int64(350)

In [10]:
print(df['clean_comment'].unique())


[' family mormon have never tried explain them they still stare puzzled from time time like some kind strange creature nonetheless they have come admire for the patience calmness equanimity acceptance and compassion have developed all the things buddhism teaches '
 'buddhism has very much lot compatible with christianity especially considering that sin and suffering are almost the same thing suffering caused wanting things shouldn want going about getting things the wrong way christian this would mean wanting things that don coincide with god will and wanting things that coincide but without the aid jesus buddhism could also seen proof god all mighty will and omnipotence certainly christians are lucky have one such christ there side but what about everyone else well many christians believe god grace salvation and buddhism god way showing grace upon others would also help study the things jesus said and see how buddha has made similar claims such rich man getting into heaven joke basica

In [11]:
df['clean_comment'].nunique()

36799

In [12]:
df.shape

(37149, 2)

In [13]:
df['clean_comment'].duplicated().sum()

np.int64(350)

In [14]:
df = df.drop_duplicates(subset=['clean_comment'])


In [15]:
df['clean_comment'].duplicated().sum()

np.int64(0)

In [16]:
df.sample(10)

,clean_comment,category
20767,wow another example abuse power these politici...,-1
28469,please let kejriwal the prime minister the fut...,0
14289,according the best political ever made lli bnj...,1
33703,noooo,0
4089,bjp only leading,0
3896,arnab kind sad ndtv jubilant,1
9206,this great step modi some highlights 500 and 1...,-1
12144,bhopal doctors submit mass resignations after...,-1
18291,newcomer his winning match the shots ishqiya,1
32080,this something called cognitive dissonance and...,1


In [17]:
# df['Sentiment'] 

In [18]:
# df['Text'] 

## Preprocessing


In [19]:
def clean_text(text):
    text = str(text).lower()                     # lowercase
    text = re.sub(r'[^\w\s]', '', text)         # remove punctuation
    text = re.sub(r'\d+', '', text)             # remove digits
    text = re.sub(r'\s+', ' ', text).strip()    # collapse multiple spaces
    return text

df['clean_text'] = df['clean_comment'].apply(clean_text)

In [20]:
df = df.copy()  # make sure df is an independent DataFrame
df['clean_text'] = df['clean_comment'].apply(clean_text)

In [ ]:
df['clean_text']

## Encodes labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
# df['label'] = label_encoder.fit_transform(df['Sentiment'])

In [ ]:
# Map numeric labels to readable names
label_map = {
    1: 'Positive',
    0: 'Neutral',
    -1: 'Negative'
}

In [ ]:
# df['label']

In [ ]:
X = df['clean_text']
y = df['category']

## test split portion

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
# X = df['clean_text']
# y = df['label']

# # Use stratification to keep class distribution similar across splits
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
# print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

### Tfidf


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

In [ ]:
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## Model creating


In [ ]:
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

In [ ]:
# sample_text = "I really enjoyed the movie!"
# cleaned = clean_text(sample_text)
# vec = vectorizer.transform([cleaned])
# pred = model.predict(vec)[0]
# print(f"Predicted label: {pred} -> {label_names[pred]}")